# Same-location site timelines — **interactive (Plotly)**

**Unique site = `client_id` + `site_id`.** Sites sharing the **exact same lat/lon**
(from `same_location_sites.csv`) are the *same physical car wash* under different operators over time.

Monthly **total washes** (`mem_wash_count + ret_wash_count`) with **month/year on the x-axis** and
**hover values** (total, membership, retail, operational_start). Two tools:
- `plot_location(gid)` — interactive chart for one location.
- an **all-110** figure with a **fixed dropdown** on top and the chart below.

In [1]:
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go

DATA = "../data"
main = pd.read_csv(f"{DATA}/main-data-6yr.csv", low_memory=False)
loc  = pd.read_csv(f"{DATA}/same_location_sites.csv")

main["total_washes"] = main.mem_wash_count.fillna(0) + main.ret_wash_count.fillna(0)
main["date"]         = pd.to_datetime(dict(year=main.year, month=main.month, day=1))

m = main.merge(loc[["loc_group","ll_key","dup_type","client_id","site_id"]],
               on=["client_id","site_id"], how="inner")
m["site"] = m.client_name + " (" + m.client_id + "|" + m.site_id.astype(str) + ")"
LOCS = sorted(m.loc_group.unique())
print(f"{len(m):,} monthly rows · {m.groupby(['client_id','site_id']).ngroups} sites · {len(LOCS)} shared locations")

5,677 monthly rows · 230 sites · 110 shared locations


In [2]:
# per-site timeline: when it started/ended, months, lifetime washes, operational_start
g = m.groupby(["loc_group","ll_key","dup_type","site"], sort=False)
timeline = (g.agg(first=("date","min"), last=("date","max"), months=("date","nunique"),
                  total_washes=("total_washes","sum"), op_start=("operational_start","first"))
              .reset_index().sort_values(["loc_group","first"]))
timeline["first"] = timeline["first"].dt.strftime("%Y-%m")
timeline["last"]  = timeline["last"].dt.strftime("%Y-%m")
timeline

,loc_group,ll_key,dup_type,site,first,last,months,total_washes,op_start
86,1,"26.1943,-80.2862",cross_operator,Sonnys Demo (demo_client|14),2023-11,2026-04,27,258,11-2023
87,1,"26.1943,-80.2862",cross_operator,Sonnys Demo (demo_client|15),2024-02,2025-06,17,996,02-2024
55,1,"26.1943,-80.2862",cross_operator,Client Design Services (cdsdemo_000777|1),2025-06,2026-05,12,1765,06-2025
88,1,"26.1943,-80.2862",cross_operator,Sonnys Demo (demo_client|16),2025-12,2026-04,3,13,12-2025
89,1,"26.1943,-80.2862",cross_operator,Sonnys Demo (demo_client|20),2026-04,2026-04,1,1,04-2026
...,...,...,...,...,...,...,...,...,...
195,108,"43.1134,-79.1235",cross_operator,Valet Car Wash (valetcw_000706|1),2025-02,2026-05,16,171076,02-2025
180,109,"43.3941,-80.3245",cross_operator,Smitty's Car Wash (smittyscw_000489|13),2024-08,2025-04,9,73791,08-2024
190,109,"43.3941,-80.3245",cross_operator,Swift Car Wash (swiftcw_000762|1),2025-05,2026-05,13,112775,05-2025
175,110,"44.4322,-73.2116",same_operator,Seaway Car Wash (seawaycarwash_000181|1),2020-01,2026-05,77,372721,01-2020


In [3]:
# interactive chart for ONE location (hover = total/mem/ret/op_start, x = month-year)
def plot_location(gid):
    d = m[m.loc_group == gid].sort_values("date")
    fig = px.line(d, x="date", y="total_washes", color="site", markers=True,
                  hover_data={"date":"|%b %Y", "total_washes":":,",
                              "mem_wash_count":":,", "ret_wash_count":":,",
                              "operational_start":True, "site":False},
                  title=f"loc {gid} · {d.ll_key.iloc[0]} · {d.state.iloc[0]} — {d.site.nunique()} operators")
    fig.update_xaxes(title="", dtick="M3", tickformat="%b %Y")
    fig.update_yaxes(title="monthly total washes")
    fig.update_layout(hovermode="x unified", legend_title="operator (client_id|site_id)", height=460)
    return fig

plot_location(2)   # Wash City -> Wash and Go -> Zips -> El Car Wash

In [4]:
# ── ALL 110 locations · native dropdown FIXED (sticky) on top, chart below — self-contained, persists on reopen ──
import json as _json
from IPython.display import HTML
pal = px.colors.qualitative.Plotly

fig = go.Figure(); trace_loc = []
for gid in LOCS:
    d = m[m.loc_group == gid].sort_values("date")
    for i, s in enumerate(dict.fromkeys(d.site)):
        ds = d[d.site == s]
        fig.add_trace(go.Scatter(
            x=ds.date, y=ds.total_washes, mode="lines+markers", name=s.split(" (")[0],
            line=dict(color=pal[i % len(pal)]), visible=(gid == LOCS[0]),
            customdata=ds[["mem_wash_count", "ret_wash_count", "operational_start"]].to_numpy(),
            hovertemplate="<b>%{fullData.name}</b><br>%{x|%b %Y}<br>total %{y:,}"
                          "<br>mem %{customdata[0]:,.0f} · ret %{customdata[1]:,.0f}"
                          "<br>op start %{customdata[2]}<extra></extra>"))
        trace_loc.append(gid)
trace_loc = np.array(trace_loc)

def _title(gid):
    dd = m[m.loc_group == gid]
    return f"loc {gid} · {dd.ll_key.iloc[0]} · {dd.state.iloc[0]} — {dd.site.nunique()} operators"

fig.update_layout(title=_title(LOCS[0]), hovermode="x unified", height=470, margin=dict(t=50, b=70),
                  legend=dict(title="operator", orientation="h", yanchor="top", y=-0.22, x=0))
fig.update_xaxes(tickformat="%b %Y", dtick="M3"); fig.update_yaxes(title="monthly total washes")

masks  = {int(g): (trace_loc == g).tolist() for g in LOCS}
titles = {int(g): _title(g) for g in LOCS}
opts   = "".join(f'<option value="{g}">loc {g} · {m[m.loc_group==g].state.iloc[0]} · {m[m.loc_group==g].site.nunique()}op</option>' for g in LOCS)
div    = fig.to_html(include_plotlyjs="cdn", full_html=False, div_id="alllocPLOT")

HTML(f'''
<div style="position:sticky;top:0;z-index:20;background:#fff;padding:8px 2px;border-bottom:1px solid #eee">
  <label style="font:600 13px sans-serif">📍 Location:
    <select id="locpick" style="font-size:13px;padding:3px 6px;max-width:460px">{opts}</select>
  </label>
</div>
{div}
<script>
(function(){{
  const M = {_json.dumps(masks)}, T = {_json.dumps(titles)};
  const sel = document.getElementById("locpick");
  sel.addEventListener("change", function(){{
    const g = sel.value;
    Plotly.restyle("alllocPLOT", {{"visible": M[g]}});
    Plotly.relayout("alllocPLOT", {{"title.text": T[g]}});
  }});
}})();
</script>
''')
